# Appendix — AWS Strands Agents: the same triage, model-driven and gated

[AWS Strands Agents](https://strandsagents.com/) is, in its own words, a "model-driven, open source" SDK for building "production-ready AI agents in a few lines of code." This appendix runs the course's fixed ticket — **TKT-2205** — through it: same tools, same world data, same economics you have hit in every other appendix. The point is not a new agent idea. It is to see Strands' `Agent`, the `@tool` decorator, and its hook-based approval gate line up, one for one, against the machinery you built by hand in Parts 1-3.

> **Before running this notebook** — Strands installs into its **own** virtual environment with its **own** Jupyter kernel. `strands-agents` constrains `mcp<2` (among other pins) in a combination the resolver proves incompatible with the course's main stack, so it cannot share the course venv. Set it up once:
>
> ```
> python -m venv .venv-strands && . .venv-strands/bin/activate
> pip install -e ".[strands]"
> python -m ipykernel install --user --name agentic-lab-strands
> ```
>
> Then, in Jupyter, pick the kernel **agentic-lab-strands** for THIS notebook (Kernel -> Change Kernel). Every cell below runs in that venv — `strands-agents`, plus the same `shoplab`, `litellm`, and `diskcache` you already know.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

## The invariant: what TKT-2205 must decide

Every framework appendix ends at the same numbers so they are comparable. The gold is not hand-typed — it comes straight from `shoplab.rules.decide`, the deterministic cascade the whole world is graded against. For TKT-2205 (an *opened* boot returned by a *non-vip member*, 18 days out, inside the 30-day window) branch 9 fires: item value ($189.99) minus a 10% restocking fee.

In [ ]:
# The one fixed decision this appendix must land on (authoritative).
from shoplab import world
from shoplab.rules import decide

orders = {o["order_id"]: o for o in world.load_orders()}
customers = {c["customer_id"]: c for c in world.load_customers()}
ticket = next(t for t in world.load_tickets()["train"] if t["ticket_id"] == "TKT-2205")
order, customer = orders[ticket["order_id"]], customers[ticket["customer_id"]]

GOLD = decide(ticket, order, customer)
print("TKT-2205 gold:", GOLD)          # partial_refund / pol-restocking / 170.99
print("arithmetic:", 189.99, "x 0.90 =", round(189.99 * 0.90, 2))

## Wiring the course model into Strands

Strands is *model-driven*: the loop is the model deciding, turn by turn, whether to call a tool or answer. It reaches the model through provider adapters, and `strands.models.litellm.LiteLLMModel` takes the same provider-prefixed string you have used since ch01. So `openrouter/deepseek/deepseek-v3.2` drops straight in as `model_id`, `params={"temperature": TEMPERATURE}` holds the course at temperature 0, and because the call still goes through `litellm` the disk cache from the config cell makes reruns ~free. `callback_handler=None` (set on the `Agent` below) just silences Strands' default token streaming so the notebook output stays tidy.

In [ ]:
from strands.models.litellm import LiteLLMModel

model = LiteLLMModel(
    client_args={"api_key": os.environ["OPENROUTER_API_KEY"]},
    model_id=MODEL,
    params={"temperature": TEMPERATURE},
)
print("model:", MODEL)

## The four ops-desk tools, as `@tool`

Strands' `@tool` decorator turns a plain function's signature and docstring into a tool schema — exactly the job `to_openai_tools` did by hand in ch02. The wrappers stay thin: each one calls straight into the `shoplab` tool it stands for, so the world data and the risky-tool `Ledger` are the same objects the rest of the course uses.

In [ ]:
import json
from strands import tool
from shoplab.tools import standard_tools, Ledger

ledger = Ledger()            # ch02's risky-tool audit log
_t = standard_tools(ledger)  # the 9 course tools, backed by the world data

@tool
def get_order(order_id: str) -> str:
    """Look up an order by id: items, totals, status, dates."""
    return json.dumps(_t["get_order"].fn(order_id))

@tool
def search_policy(query: str, k: int = 2) -> str:
    """Keyword-search the 12 store policy documents."""
    return json.dumps(world.search_policy(query, k))

@tool
def calc(expr: str) -> str:
    """Evaluate an arithmetic expression, e.g. '0.9 * 189.99'."""
    return json.dumps(_t["calc"].fn(expr))

In [ ]:
from strands.hooks import BeforeToolCallEvent

@tool
def issue_refund(order_id: str, amount_usd: float, reason: str) -> str:
    """Send money back to the customer. Irreversible."""
    return json.dumps(_t["issue_refund"].fn(order_id, amount_usd, reason))

# The ch08 approval gate, as a Strands hook. BeforeToolCallEvent fires just
# before any tool runs; setting event.cancel_tool cancels THIS call and hands
# the model an error result instead of executing it. We cancel issue_refund
# until a human approves, capturing the exact pending call so approval can
# execute it verbatim.
APPROVAL = {"granted": False, "pending": []}

def approval_gate(event: BeforeToolCallEvent):
    tu = event.tool_use
    if tu.get("name") == "issue_refund" and not APPROVAL["granted"]:
        APPROVAL["pending"].append(tu.get("input"))
        event.cancel_tool = ("Approval required: a human must approve this refund "
                             "before it executes. Do not retry; approval is pending.")

TOOLS = [get_order, search_policy, calc, issue_refund]

## Run the triage — the gate holds the refund

Calling the `Agent` runs Strands' model-driven loop: model call, tool calls, repeat, until the agent answers. When it reaches `issue_refund`, the hook's `cancel_tool` stops that call before any money moves and returns an "approval pending" result the model reads. The `Ledger` stays empty; `APPROVAL["pending"]` holds the exact call that was blocked.

In [ ]:
from strands import Agent

SYSTEM = (
    "You are the Larkspur Outfitters ops-desk agent. Triage one return ticket end to end "
    "with your tools, in order: get_order for the unit price, search_policy for the governing "
    "policy, calc the refund (item value minus a 10% restocking fee, rounded to the nearest "
    "cent), then issue_refund for that exact amount. Then state the decision, the policy id, "
    "and the dollar amount."
)
TASK = (
    "Ticket TKT-2205: order ORD-7312, customer CUST-07 (member, non-vip), sku LK-1016 qty 1, "
    "condition opened, 18 days since delivery, requests a refund to the original payment method."
)
agent = Agent(model=model, tools=TOOLS, system_prompt=SYSTEM, callback_handler=None)
agent.hooks.add_callback(BeforeToolCallEvent, approval_gate)

result = agent(TASK)
for call in APPROVAL["pending"]:
    print("APPROVAL NEEDED -> issue_refund", call)
print("ledger (has the refund executed?):", ledger.entries)

> **What you should see:** one pending `issue_refund` call, its `amount_usd` already **170.99**, and the ledger **still empty**. The gate stopped the loop the instant the agent reached for money — before any of it moved. The model computed 170.99 itself: $189.99 minus the 10% opened-item restocking fee.

## Approve, execute, and land

A human approves the pending call. As in ch08's `require_approval`, approval executes the **exact captured call** — not a fresh model turn that might drift — so the amount that moves is provably the one the gate showed. `issue_refund` now runs against the real `shoplab` tool, the `Ledger` records it, and we check it against the gold.

In [ ]:
APPROVAL["granted"] = True           # a human says yes
pending = APPROVAL["pending"][-1]
_t["issue_refund"].fn(**pending)     # execute the approved call verbatim

print("ledger:", ledger.entries)
moved = ledger.entries[-1]["amount_usd"]
print(f"agent moved ${moved}  |  gold ${GOLD['refund_usd']}  |  match: {moved == GOLD['refund_usd']}")
print("decision:", GOLD["decision"], "|", GOLD["policy_id"])

> **What you should see:** after approval the refund executes; the ledger holds a single `issue_refund` entry for **170.99**, matching the gold exactly — the same **partial_refund / pol-restocking / 170.99** every appendix lands on. The decision and policy come from the same cascade (`shoplab.rules.decide`) the agent had to reconstruct with its tools; the dollar amount it moved is provably the one the gate surfaced.

## Machinery map: Strands concept to the part you built

Line them up and Strands stops being magic — it is Parts 1-3, packaged with defaults and a runtime. Nothing here is a new idea about agents; it is the ops-desk loop you already wrote, wearing Strands' names.

| AWS Strands Agents | Your hand-built equivalent | Built in |
|---|---|---|
| `Agent(...)` call — the loop | `run_agent`: model call -> execute tool calls -> repeat until done | ch02 |
| model-driven action selection | the model-driven archetype (the model picks each next action) | ch02 / ch15 |
| `@tool` (signature + docstring -> schema) | `to_openai_tools` over the `Tool` registry | ch02 |
| `LiteLLMModel(model_id=MODEL)` | `shoplab.llm.complete` over LiteLLM, same provider string | ch01 |
| a tool returning an `{"error": ...}` string the model reads | `run_tool` swallowing exceptions into an observation | ch02 |
| `BeforeToolCallEvent` + `cancel_tool` | `require_approval` wrapping a risky `Tool` in a gate | ch08 |
| `Agent.messages` retained across turns | the message list you threaded through the loop by hand | ch02 |

## The honest read

Strands gave us three things for free that we built by hand: the model-driven loop (`Agent`), the schema plumbing (`@tool`), and a clean place to intercept a risky call (`BeforeToolCallEvent` + `cancel_tool`). We used that lightweight hook as our approval gate because it maps one-for-one onto ch08's `require_approval`; Strands also ships a heavier interrupt/intervention system for human-in-the-loop, plus sessions and multi-agent orchestration, none of which this one ticket needed. What none of it changed is the hard part — the world, the tools, the rules, and the decision. Those are still `shoplab`. An appendix, not a rewrite: the framework is a convenience over machinery you now understand well enough to have skipped it.